# 第 3 讲：数据导入、清洗与基础可视化

> 用 Python 完成数据清洗流程，输出缺失值处理、描述统计和基础图表。

## 课件导学

**任务情境**：附件数据通常带缺失值、异常值和量纲差异，本讲训练一条最小清洗与可视化链。

**关键概念**

- 缺失值识别
- 填补策略
- 描述统计
- 基础图表

**本讲产物**

- raw_health_data.csv
- descriptive_summary.csv
- basic_visualization.png

## 资料链接与阅读抓手

下面这些链接优先选官方文档或稳定资料。课前先看标题和示例，课堂中遇到参数或概念再回查。

- [pandas 缺失数据指南](https://pandas.pydata.org/docs/user_guide/missing_data.html)：学习缺失值的表示、检测和填补。
- [pandas groupby 用户指南](https://pandas.pydata.org/docs/user_guide/groupby.html)：用分组统计检查不同类别的数据质量。
- [Matplotlib pyplot 教程](https://matplotlib.org/stable/tutorials/pyplot.html)：掌握直方图、散点图和基础图形输出。

## 使用说明

- 先从上到下运行全部单元。
- 每讲都会生成一个 `outputs/lesson-xx/` 目录，用于保存图表、表格或报告片段。
- 示例数据均为教学用合成数据，真实比赛中应替换为题目附件数据。

In [ ]:
LESSON_ID = "lesson-03"

from pathlib import Path
import os
import math
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

OUTPUT_ROOT = Path(os.environ.get("COURSE_OUTPUT_DIR", REPO_ROOT / "outputs")) / LESSON_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)
print(f"Output directory: {OUTPUT_ROOT}")

## 可运行示例与讲解路线

In [ ]:
n = 160
data = pd.DataFrame({
    "age": rng.integers(18, 75, n),
    "bmi": rng.normal(25, 4, n),
    "glucose": rng.normal(110, 22, n),
    "blood_pressure": rng.normal(78, 10, n),
    "risk_group": rng.choice(["low", "medium", "high"], n, p=[0.45, 0.35, 0.20]),
})
for col in ["bmi", "glucose", "blood_pressure"]:
    miss_idx = rng.choice(data.index, size=8, replace=False)
    data.loc[miss_idx, col] = np.nan
raw_path = OUTPUT_ROOT / "raw_health_data.csv"
data.to_csv(raw_path, index=False, encoding="utf-8-sig")
data.head()

In [ ]:
clean = data.copy()
numeric_cols = clean.select_dtypes(include=np.number).columns
for col in numeric_cols:
    clean[col] = clean[col].fillna(clean[col].median())

summary = clean[numeric_cols].describe().T[["mean", "std", "min", "max"]]
summary.to_csv(OUTPUT_ROOT / "descriptive_summary.csv", encoding="utf-8-sig")
summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
clean["glucose"].hist(ax=axes[0], bins=18)
axes[0].set_title("Glucose distribution")
clean.boxplot(column="bmi", by="risk_group", ax=axes[1])
axes[1].set_title("BMI by risk group")
axes[1].figure.suptitle("")
axes[2].scatter(clean["age"], clean["glucose"], alpha=0.7)
axes[2].set_title("Age vs glucose")
for ax in axes:
    ax.set_xlabel("")
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "basic_visualization.png", dpi=160)
plt.show()

## 实战环节

**课堂任务**

- 统计每列缺失数量。
- 比较均值填补和中位数填补。
- 输出 1 张分布图、1 张箱线图、1 张散点图。

**课后挑战**：给清洗步骤补一段 200 字数据质量说明，说明为什么选择该填补策略。

**验收清单**

- 是否保存原始数据副本
- 是否说明缺失处理方式
- 图表标题和变量含义是否清楚

In [ ]:
lesson_resources = [{'title': 'pandas 缺失数据指南', 'url': 'https://pandas.pydata.org/docs/user_guide/missing_data.html', 'reading_note': '学习缺失值的表示、检测和填补。'}, {'title': 'pandas groupby 用户指南', 'url': 'https://pandas.pydata.org/docs/user_guide/groupby.html', 'reading_note': '用分组统计检查不同类别的数据质量。'}, {'title': 'Matplotlib pyplot 教程', 'url': 'https://matplotlib.org/stable/tutorials/pyplot.html', 'reading_note': '掌握直方图、散点图和基础图形输出。'}]
lesson_deliverables = ['raw_health_data.csv', 'descriptive_summary.csv', 'basic_visualization.png']
lesson_checklist = ['是否保存原始数据副本', '是否说明缺失处理方式', '图表标题和变量含义是否清楚']

pd.DataFrame(lesson_resources).to_csv(OUTPUT_ROOT / "lesson_resources.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"deliverable": lesson_deliverables}).to_csv(OUTPUT_ROOT / "lesson_deliverables.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"check_item": lesson_checklist}).to_csv(OUTPUT_ROOT / "lesson_checklist.csv", index=False, encoding="utf-8-sig")
print("Saved courseware resource map, deliverables, and checklist.")